# Purpose

## Merge + Annualize

# Imports

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Set data paths
PROJECT_ROOT = Path().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
# Import datasets
census_path = DATA_PROCESSED / "census_fl_county_year.csv"
zillow_path = DATA_PROCESSED / "zhvi_fl_long.csv"
census_df = pd.read_csv(census_path)
zillow_df = pd.read_csv(zillow_path)

# Review Datasets

In [4]:
zillow_df.shape

(4891, 12)

In [5]:
zillow_df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,date,zhvi,year
0,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-01-31,205945.224353,2020
1,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-02-29,207639.389278,2020
2,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-03-31,209186.222049,2020
3,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-04-30,210416.781373,2020
4,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-05-31,211241.441421,2020


In [6]:
zillow_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4891 entries, 0 to 4890
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RegionID           4891 non-null   int64  
 1   SizeRank           4891 non-null   int64  
 2   RegionName         4891 non-null   object 
 3   RegionType         4891 non-null   object 
 4   StateName          4891 non-null   object 
 5   State              4891 non-null   object 
 6   Metro              3723 non-null   object 
 7   StateCodeFIPS      4891 non-null   int64  
 8   MunicipalCodeFIPS  4891 non-null   int64  
 9   date               4891 non-null   object 
 10  zhvi               4891 non-null   float64
 11  year               4891 non-null   int64  
dtypes: float64(1), int64(5), object(6)
memory usage: 458.7+ KB


In [7]:
census_df.shape

(335, 7)

In [8]:
census_df.head()

,fips,STNAME,CTYNAME,year,domestic_mig,population,rdomestic_mig
0,12001,Florida,Alachua County,2020,1315,279765,NaN
1,12001,Florida,Alachua County,2021,1142,281710,4.067857
2,12001,Florida,Alachua County,2022,1268,285241,4.473050
3,12001,Florida,Alachua County,2023,408,288962,1.421100
4,12001,Florida,Alachua County,2024,-1141,291782,-3.929442


# Zillow dataset cleaning

## Convert date to datetime

In [9]:
zillow_df["date"] = pd.to_datetime(zillow_df["date"])

In [10]:
zillow_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4891 entries, 0 to 4890
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   RegionID           4891 non-null   int64         
 1   SizeRank           4891 non-null   int64         
 2   RegionName         4891 non-null   object        
 3   RegionType         4891 non-null   object        
 4   StateName          4891 non-null   object        
 5   State              4891 non-null   object        
 6   Metro              3723 non-null   object        
 7   StateCodeFIPS      4891 non-null   int64         
 8   MunicipalCodeFIPS  4891 non-null   int64         
 9   date               4891 non-null   datetime64[ns]
 10  zhvi               4891 non-null   float64       
 11  year               4891 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(5)
memory usage: 458.7+ KB


In [11]:
zillow_df["date"].min(), zillow_df["date"].max()

(Timestamp('2020-01-31 00:00:00'), Timestamp('2026-01-31 00:00:00'))

## Create FIPS (to match Census dataframe for merge)

In [12]:
zillow_df["state_fips"] = (zillow_df["StateCodeFIPS"].astype(int).astype(str).str.zfill(2))
# Converted to int first to standardize (1.0 -> 1), then back to string to use functions

In [13]:
zillow_df["county_fips"] = (zillow_df["MunicipalCodeFIPS"].astype(int).astype(str).str.zfill(3))

In [14]:
zillow_df["fips"] = zillow_df["state_fips"] + zillow_df["county_fips"]

In [15]:
zillow_df["fips"].nunique()

67

In [16]:
zillow_df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,date,zhvi,year,state_fips,county_fips,fips
0,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-01-31,205945.224353,2020,12,001,12001
1,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-02-29,207639.389278,2020,12,001,12001
2,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-03-31,209186.222049,2020,12,001,12001
3,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-04-30,210416.781373,2020,12,001,12001
4,1509,251,Alachua County,county,FL,FL,"Gainesville, FL",12,1,2020-05-31,211241.441421,2020,12,001,12001


## Annualize ZHVI

In [17]:
zillow_annual = (zillow_df.groupby(["fips", "RegionName", "year"], as_index=False)["zhvi"].mean())
# as_index=false added to keep dataframe clean of multiindex for merge

In [18]:
zillow_annual.head()

,fips,RegionName,year,zhvi
0,12001,Alachua County,2020,213860.291033
1,12001,Alachua County,2021,244364.099887
2,12001,Alachua County,2022,282232.292655
3,12001,Alachua County,2023,295677.948240
4,12001,Alachua County,2024,304504.265219


In [19]:
zillow_annual = zillow_annual.rename(columns={'zhvi': 'zhvi_annual'})

### Sort

In [20]:
zillow_annual = zillow_annual.sort_values(["fips", "year"]).reset_index(drop=True)

# Sanity checks

In [21]:
zillow_annual.shape

(469, 4)

In [22]:
zillow_annual["fips"].nunique()

67

In [23]:
zillow_annual["year"].min(), zillow_annual["year"].max()

(2020, 2026)

In [25]:
zillow_annual.head(10)

,fips,RegionName,year,zhvi_annual
0,12001,Alachua County,2020,213860.291033
1,12001,Alachua County,2021,244364.099887
2,12001,Alachua County,2022,282232.292655
3,12001,Alachua County,2023,295677.948240
4,12001,Alachua County,2024,304504.265219
5,12001,Alachua County,2025,299662.702434
6,12001,Alachua County,2026,296180.658305
7,12003,Baker County,2020,214588.831681
8,12003,Baker County,2021,250077.267178
9,12003,Baker County,2022,293861.137166


# Manage Census datatypes for merging

In [35]:
census_df.head(10)

,fips,STNAME,CTYNAME,year,domestic_mig,population,rdomestic_mig
0,12001,Florida,Alachua County,2020,1315,279765,NaN
1,12001,Florida,Alachua County,2021,1142,281710,4.067857
2,12001,Florida,Alachua County,2022,1268,285241,4.473050
3,12001,Florida,Alachua County,2023,408,288962,1.421100
4,12001,Florida,Alachua County,2024,-1141,291782,-3.929442
5,12003,Florida,Baker County,2020,-141,28122,NaN
6,12003,Florida,Baker County,2021,216,28378,7.646018
7,12003,Florida,Baker County,2022,-657,27781,-23.397853
8,12003,Florida,Baker County,2023,673,28542,23.897875
9,12003,Florida,Baker County,2024,709,29325,24.504467


In [29]:
census_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fips           335 non-null    int64  
 1   STNAME         335 non-null    object 
 2   CTYNAME        335 non-null    object 
 3   year           335 non-null    int64  
 4   domestic_mig   335 non-null    int64  
 5   population     335 non-null    int64  
 6   rdomestic_mig  268 non-null    float64
dtypes: float64(1), int64(4), object(2)
memory usage: 18.4+ KB


In [32]:
census_df["fips"] = census_df["fips"].astype(str).str.zfill(5)

In [34]:
census_df.head()

,fips,STNAME,CTYNAME,year,domestic_mig,population,rdomestic_mig
0,12001,Florida,Alachua County,2020,1315,279765,NaN
1,12001,Florida,Alachua County,2021,1142,281710,4.067857
2,12001,Florida,Alachua County,2022,1268,285241,4.473050
3,12001,Florida,Alachua County,2023,408,288962,1.421100
4,12001,Florida,Alachua County,2024,-1141,291782,-3.929442


# Merge Datasets

In [37]:
# Inner join to keep rows where both datasets exist
merged = zillow_annual.merge(census_df, on=["fips", "year"], how="inner")

In [46]:
merged.shape # Expect 335 rows (67 counties x 5 years)

(335, 9)

In [39]:
merged.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467


In [40]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fips           335 non-null    object 
 1   RegionName     335 non-null    object 
 2   year           335 non-null    int64  
 3   zhvi_annual    335 non-null    float64
 4   STNAME         335 non-null    object 
 5   CTYNAME        335 non-null    object 
 6   domestic_mig   335 non-null    int64  
 7   population     335 non-null    int64  
 8   rdomestic_mig  268 non-null    float64
dtypes: float64(2), int64(3), object(4)
memory usage: 23.7+ KB


In [45]:
merged["year"].unique() # view array of values

array([2020, 2021, 2022, 2023, 2024])

In [44]:
merged["fips"].nunique() # expect 67 counties

67

# Sort values for cleanliness

In [47]:
merged = merged.sort_values(["fips","year"]).reset_index(drop=True)

In [48]:
merged.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467


# Export Dataset

In [49]:
merged.to_csv("../data/processed/merged_fl_county_year.csv", index=False)